# 01 — Exploration des données · Databricks
**Projet** : FakeNews Analyzer
**Exécuter après** : upload CSV sur `dbfs:/FileStore/fakenews/raw/`
---

In [ ]:
%pip install wordcloud vaderSentiment seaborn -q

In [ ]:
# ============================================================
# Setup Databricks : résolution du repo root + chemins DBFS
# ============================================================
import sys, os

_ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_parts = _ctx.split('/')
_idx   = next((i for i, p in enumerate(_parts) if p == 'fakenews-analyzer'), None)
_REPO  = '/Workspace/' + '/'.join(_parts[1:_idx+1]) if _idx else '/Workspace/Repos'
if _REPO not in sys.path:
    sys.path.insert(0, _REPO)

from spark_utils import (
    load_raw_sources, show_label_distribution,
    save_parquet, load_parquet,
    stratified_split, check_class_balance
)

RAW_DIR       = '/dbfs/FileStore/fakenews/raw'
PROCESSED_DIR = '/dbfs/FileStore/fakenews/processed'
MODELS_DIR    = '/dbfs/FileStore/fakenews/models'
COLAB_DIR     = '/dbfs/FileStore/fakenews/colab_exports'
REPORTS_DIR   = '/dbfs/FileStore/fakenews/reports'

for d in [PROCESSED_DIR, MODELS_DIR, COLAB_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Repo root  : {_REPO}')
print('Chemins DBFS OK')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
plt.style.use('dark_background')
print('Imports OK')

## Section 1 — Chargement des sources

In [ ]:
# spark est fourni par le cluster — pas besoin de get_spark_session()
sources = load_raw_sources(spark, RAW_DIR)
print(f'Sources chargees : {list(sources.keys())}')

## Section 2 — Stats de base

In [ ]:
for name, df in sources.items():
    print(f'\n{"="*50}')
    print(f'Source : {name}  |  Lignes : {df.count():,}  |  Colonnes : {len(df.columns)}')
    df.printSchema()
    display(df.limit(3))

## Section 3 — Distribution des labels

In [ ]:
from pyspark.sql import functions as F

for name, df in sources.items():
    label_candidates = [c for c in df.columns if c.lower() in ('label', 'fraudulent', 'fake')]
    if not label_candidates:
        print(f'[{name}] Pas de colonne label detectee')
        continue
    label_col = label_candidates[0]
    print(f'\n[{name}] — colonne label : "{label_col}"')
    show_label_distribution(df, label_col)

In [ ]:
fig, axes = plt.subplots(1, max(len(sources), 1), figsize=(5 * len(sources), 5))
if len(sources) == 1:
    axes = [axes]
for ax, (name, df) in zip(axes, sources.items()):
    label_candidates = [c for c in df.columns if c.lower() in ('label', 'fraudulent')]
    if not label_candidates:
        continue
    label_col = label_candidates[0]
    counts = df.groupBy(label_col).count().toPandas()
    counts['label_name'] = counts[label_col].apply(lambda x: 'FAKE' if str(x) in ('1','FAKE','fake') else 'REAL')
    colors = ['#ff4444' if l == 'FAKE' else '#00ccaa' for l in counts['label_name']]
    ax.bar(counts['label_name'], counts['count'], color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(f'Distribution — {name}', fontsize=12)
    ax.set_ylabel("Nombre d'articles")
    for bar, cnt in zip(ax.patches, counts['count']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{cnt:,}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, '01_label_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — Longueurs de texte

In [ ]:
for name, df in sources.items():
    text_col = next((c for c in df.columns if c.lower() in ('text','statement','tweet','content')), None)
    if not text_col:
        continue
    df_len = df.withColumn('text_len', F.length(F.col(text_col))).select('text_len')
    stats = df_len.describe().toPandas()
    print(f'\n[{name}] Longueurs de texte (caracteres) :')
    display(stats)

## Section 5 — Top 20 mots FAKE vs REAL

In [ ]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover
from pyspark.ml import Pipeline

STOPWORDS = StopWordsRemover.loadDefaultStopWords('english')

def top_words(df, text_col, label_col, label_value, n=20):
    from pyspark.sql.functions import explode, col
    subset    = df.filter(F.col(label_col).cast('string') == str(label_value))
    tokenizer = RegexTokenizer(inputCol=text_col, outputCol='tokens_raw', pattern=r'\W+', minTokenLength=3)
    remover   = StopWordsRemover(inputCol='tokens_raw', outputCol='tokens', stopWords=STOPWORDS)
    df_tok    = Pipeline(stages=[tokenizer, remover]).fit(subset).transform(subset)
    return (df_tok.select(explode(col('tokens')).alias('word'))
            .groupBy('word').count().orderBy('count', ascending=False).limit(n).toPandas())

for name, df in sources.items():
    text_col  = next((c for c in df.columns if c.lower() in ('text','statement')), None)
    label_col = next((c for c in df.columns if c.lower() in ('label','fraudulent')), None)
    if not text_col or not label_col:
        continue
    fake_words = top_words(df, text_col, label_col, 1)
    real_words = top_words(df, text_col, label_col, 0)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.barh(fake_words['word'][::-1], fake_words['count'][::-1], color='#ff4444')
    ax1.set_title('Top 20 mots — FAKE')
    ax2.barh(real_words['word'][::-1], real_words['count'][::-1], color='#00ccaa')
    ax2.set_title('Top 20 mots — REAL')
    plt.tight_layout()
    plt.show()
    break

## Section 6 — Rapport de qualité

In [ ]:
MIN_TEXT_LENGTH = 20
quality_report = []
for name, df in sources.items():
    text_col = next((c for c in df.columns if c.lower() in ('text','statement','tweet')), None)
    if not text_col:
        continue
    total     = df.count()
    nulls     = df.filter(F.col(text_col).isNull()).count()
    empty     = df.filter(F.trim(F.col(text_col)) == '').count()
    too_short = df.filter(F.length(F.col(text_col)) < MIN_TEXT_LENGTH).count()
    quality_report.append({
        'source': name, 'total': total, 'nulls': nulls, 'empty': empty,
        'too_short': too_short, 'exploitable': total - nulls - empty,
        'qualite_%': round((total - nulls - empty) / total * 100, 1) if total > 0 else 0
    })
report_df = pd.DataFrame(quality_report)
display(report_df)
report_df.to_csv(os.path.join(REPORTS_DIR, '01_quality_report.csv'), index=False)
print(f'Rapport sauvegarde : {REPORTS_DIR}/01_quality_report.csv')
print('Prochaine etape : 02_cleaning_db.ipynb')